# 🎤 YingMusic-SVC — Google Colab (Gradio)

1. `Runtime → Change runtime type → T4 GPU`
2. กด ▶ ที่ Cell ด้านล่าง (Cell เดียว: ติดตั้ง → โหลดโมเดล → เปิด Gradio)
3. รอจนขึ้น `Running on public URL: https://xxxx.gradio.live` → คลิกเปิด → อัปโหลดเพลง + เสียงเป้าหมาย → กด **Process All**

ครั้งแรกใช้เวลาติดตั้ง + โหลดโมเดลประมาณ 5–8 นาที · Cell จะรันค้างไว้ตลอดระหว่างใช้งาน (ถ้ากด ⏹ ลิงก์จะปิด)

In [ ]:
#@title ▶ ติดตั้ง + เปิด Gradio
REPO_URL = "https://github.com/DDME36/YingMusic-SVC"  #@param {type:"string"}
BRANCH = ""  #@param {type:"string"}
#@markdown ใส่ `HF_TOKEN` เฉพาะกรณีที่ Hugging Face ขอ login
HF_TOKEN = ""  #@param {type:"string"}

import os, sys, subprocess
if subprocess.run(["nvidia-smi", "-L"], capture_output=True).returncode != 0:
    raise SystemExit("❌ ไม่พบ GPU — ไปที่ Runtime → Change runtime type → เลือก T4 GPU แล้วรันใหม่")

# --- clone ---
REPO_DIR = "/content/YingMusic-SVC"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    branch_flag = f"-b {BRANCH}" if BRANCH else ""
    !git clone -q --depth 1 $branch_flag $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR pull -q --ff-only  # อัปเดตโค้ดถ้าเคย clone ไว้แล้ว
%cd $REPO_DIR
!git log -1 --format="code version: %h %s"

# --- dependencies ---
# ใช้ torch/torchaudio ที่ Colab ติดตั้งมาแล้ว (ล็อกเวอร์ชันไว้ไม่ให้ pip เปลี่ยน)
# sox (binary) ใช้ทำ echo/reverb ตอนมิกซ์กลับ แทน torchaudio.sox_effects ที่ถูกตัดออกใน torchaudio รุ่นใหม่
print("กำลังติดตั้ง dependencies ...")
!apt-get -qq update > /dev/null && apt-get -qq install -y sox libsox-fmt-all ffmpeg > /dev/null
from importlib.metadata import version
TORCH_PINS = f"torch=={version('torch')} torchaudio=={version('torchaudio')}"
!pip install -q $TORCH_PINS \
    "librosa>=0.10.2,<1" "transformers>=4.46.3,<5" "huggingface-hub>=0.28.1,<1.0" \
    munch einops "descript-audio-codec==1.0.0" "gradio>=5.23,<6" pydub soundfile pyyaml \
    omegaconf ml_collections loralib wandb beartype "rotary_embedding_torch==0.3.5" matplotlib

# --- models ---
print("กำลังโหลดโมเดล ...")
from huggingface_hub import hf_hub_download, snapshot_download
token = HF_TOKEN or None
SVC_CKPT = os.path.abspath(hf_hub_download("GiantAILab/YingMusic-SVC", "YingMusic-SVC-full.pt",
                                           local_dir="checkpoints", token=token))
hf_hub_download("GiantAILab/YingMusic-SVC", "bs_roformer.ckpt",
                local_dir="accom_separation/ckpt/bs_roformer", token=token)
# โมเดลเสริม: โหลดล่วงหน้าเข้า cache path เดียวกับที่ my_inference.py / hf_utils.py ใช้
hf_hub_download("lj1995/VoiceConversionWebUI", "rmvpe.pt", cache_dir="./checkpoints")
hf_hub_download("funasr/campplus", "campplus_cn_common.bin", cache_dir="./checkpoints")
snapshot_download("nvidia/bigvgan_v2_44khz_128band_512x", cache_dir="./checkpoints/hf_cache",
                  allow_patterns=["config.json", "bigvgan_generator.pt"])
snapshot_download("openai/whisper-small", cache_dir="./checkpoints/hf_cache",
                  allow_patterns=["*.json", "*.safetensors", "*.txt"])

os.environ["YINGMUSIC_SVC_CKPT"] = SVC_CKPT
!python -c "import torch, torchaudio, librosa, gradio, transformers; print('torch', torch.__version__, '| torchaudio', torchaudio.__version__, '| gradio', gradio.__version__, '| CUDA', torch.cuda.is_available())"

# --- launch ---
print("✅ ติดตั้งเสร็จ — กำลังเปิด Gradio (รอลิงก์ gradio.live ด้านล่าง)")
!PYTHONUNBUFFERED=1 python -u -c "import gradio_app as g; g.build_ui().queue().launch(share=True, server_name='0.0.0.0', server_port=7860)"

### วิธีใช้ใน Gradio
- **⚡ One-Click Separation + SVC** — อัปโหลดเพลงเต็ม (มีดนตรี) + เสียงเป้าหมาย → กด *Process All* ได้ทั้งเสียงร้องที่แปลงแล้วและเพลงที่มิกซ์กลับ
- **🎤 Standalone SVC** — ถ้ามีไฟล์เสียงร้องล้วนอยู่แล้ว
- **🎚️ Accompaniment Separation** — แยกเสียงร้อง/ดนตรีอย่างเดียว

เสียงเป้าหมายควรเป็นเสียงร้อง/พูดที่สะอาด ไม่มีดนตรี ยาว 10–25 วินาที (หรือกด *Use Sample Timbre*) · กด Convert ครั้งแรกจะช้า (~1 นาที, โหลดโมเดล) ครั้งต่อไปเร็วขึ้น